In [ ]:
from pathlib import Path
import csv
import hashlib
import hmac
import math
import os
import re
import secrets
import tempfile
from datetime import datetime
from decimal import Decimal, InvalidOperation
from getpass import getpass

CSV_FIELDS = ["transaction_id", "date", "category", "description", "amount"]
SAMPLE_CSV = Path("MIT103_A3_Revised_T3_2026_Sample_Transactions.csv")
WORKING_CSV = Path("my_transactions.csv")

# COLAB PERSISTENCE OPTION: To keep the working CSV in Google Drive, run:
# from google.colab import drive
# drive.mount('/content/drive')
# WORKING_CSV = Path('/content/drive/MyDrive/MIT103_A3/my_transactions.csv')
# Keep the sample CSV in the runtime Files panel; never edit it.
print("Sample CSV found:", SAMPLE_CSV.exists())
print("Working CSV:", WORKING_CSV)


Sample CSV found: False
Working CSV: my_transactions.csv


## 1 — Class design
`Transaction` keeps the Phase 1 attributes and adds an ID. It validates dates, essential text and a signed amount in dollars.

In [ ]:

from pathlib import Path

csv_text = """transaction_id,date,category,description,amount
T001,02/11/2026,Food,Groceries,-48.50
T002,03/11/2026,Transport,Train card,-25.00
T003,04/11/2026,Income,Casual shift,180.00
T004,05/11/2026,Food,Lunch,-14.20
T005,06/11/2026,Bills,Mobile plan,-35.00
T006,07/11/2026,Transport,Bus ticket,-5.60
T007,08/11/2026,Income,Refund,20.00
T008,09/11/2026,Food,Coffee,-6.50
"""

filename = Path(
    "MIT103_A3_Revised_T3_2026_Sample_Transactions.csv"
)

if not filename.exists():
    filename.write_text(csv_text, encoding="utf-8")

print("Sample CSV found:", filename.exists())
print("Working CSV:", "my_transactions.csv")


Sample CSV found: True
Working CSV: my_transactions.csv


In [ ]:
class Transaction:
    """One validated income or expense record (positive income, negative expense)."""

    def __init__(self, transaction_id, date, category, description, amount):
        transaction_id = str(transaction_id).strip().upper()
        date = str(date).strip()
        category = str(category).strip()
        description = str(description).strip()

        if not re.fullmatch(r"T\d{3,}", transaction_id):
            raise ValueError("Transaction ID must look like T001 or T009.")
        if not re.fullmatch(r"\d{2}/\d{2}/\d{4}", date):
            raise ValueError("Date must use DD/MM/YYYY.")
        try:
            datetime.strptime(date, "%d/%m/%Y")
        except ValueError as exc:
            raise ValueError("Date is not a real calendar date.") from exc
        if not category or not description:
            raise ValueError("Category and description cannot be empty.")
        if len(category) > 60 or len(description) > 120:
            raise ValueError("Category or description is too long.")
        try:
            money = Decimal(str(amount).strip())
        except (InvalidOperation, ValueError) as exc:
            raise ValueError("Amount must be a number, e.g. -12.50 or 200.00.") from exc
        if not money.is_finite():
            raise ValueError("Amount cannot be NaN or infinity.")
        if money == 0:
            raise ValueError("Amount cannot be zero: use positive income or negative expense.")
        if money != money.quantize(Decimal("0.01")):
            raise ValueError("Amount must have no more than two decimal places.")
        if abs(money) > Decimal("1000000000"):
            raise ValueError("Amount is outside the allowed demonstration range.")

        self.transaction_id = transaction_id
        self.date = date
        self.category = category
        self.description = description
        self.amount = float(money)

    def __str__(self):
        return (f"{self.transaction_id} | {self.date} | {self.category} | "
                f"{self.description} | ${self.amount:+.2f}")

    def to_row(self):
        return {"transaction_id": self.transaction_id, "date": self.date,
                "category": self.category, "description": self.description,
                "amount": f"{self.amount:.2f}"}


## 2 — Role permissions and core operations
Both `add_transaction()` and `delete_transaction()` check the role *inside the function*. Display/Search are available to both authenticated roles.

In [ ]:
def is_authorised(role, action):
    """Central allow-list. Unknown roles and actions never gain permission."""
    permissions = {
        "admin": {"add", "display", "search", "delete", "totals"},
        "viewer": {"display", "search", "totals"},
    }
    return action in permissions.get(str(role).lower(), set())


In [ ]:
def add_transaction(transactions, transaction_id, date, category, description, amount, role):
    """Reject Viewer/unknown-role calls even if someone bypasses the menu."""
    if not is_authorised(role, "add"):
        print(f"{role}: access denied for Add.")
        return False
    try:
        record = Transaction(transaction_id, date, category, description, amount)
    except ValueError as exc:
        print(f"Input error: {exc}")
        return False
    if any(existing.transaction_id == record.transaction_id for existing in transactions):
        print(f"Input error: duplicate transaction ID {record.transaction_id}.")
        return False
    transactions.append(record)
    print(f"Admin successfully added: {record.transaction_id}")
    return True


def display_transactions(transactions):
    if not transactions:
        print("No transactions to display.")
        return
    print("\n--- Stored Transactions ---")
    for number, transaction in enumerate(transactions, start=1):
        print(f"{number:>2}. {transaction}")
    print(f"Total records: {len(transactions)}")


def search_transactions(transactions, query):
    query = str(query).strip()
    if not query:
        print("Search error: enter an ID or a category.")
        return []
    matches = [item for item in transactions
               if item.transaction_id.upper() == query.upper()
               or query.casefold() in item.category.casefold()]
    if matches:
        print(f"Found {len(matches)} result(s):")
        for item in matches:
            print(item)
    else:
        print("No matching transaction found.")
    return matches


def delete_transaction(transactions, transaction_id, role):
    """The role check is inside the function, not just in the menu."""
    if not is_authorised(role, "delete"):
        print(f"{role}: access denied for Delete.")
        return False
    requested_id = str(transaction_id).strip().upper()
    for position, transaction in enumerate(transactions):
        if transaction.transaction_id == requested_id:
            removed = transactions.pop(position)
            print(f"Deleted {removed.transaction_id}.")
            return True
    print(f"Delete failed: transaction {requested_id} was not found.")
    return False


def category_totals(transactions):
    """Dictionary combines amounts under each category in one pass."""
    totals = {}
    for transaction in transactions:
        totals[transaction.category] = (totals.get(transaction.category, 0.0)
                                        + transaction.amount)
    return {category: round(value, 2) for category, value in totals.items()}


def display_category_totals(transactions):
    print("\n--- Category Totals ($) ---")
    for category, total in sorted(category_totals(transactions).items()):
        print(f"{category}: {total:+.2f}")
    if not transactions:
        print("No transactions recorded.")
    print(f"Net balance: ${sum(item.amount for item in transactions):+.2f}")


## 3 — CSV handling and restoration
Use `csv.DictReader` to load five fixed headings. `save_transactions` writes atomically to a separate file. Running `initialise_working_data` again loads existing edits instead of replacing them with the sample.

In [ ]:
def load_transactions(filename):
    """Read valid rows using DictReader; report and skip damaged rows."""
    filename = Path(filename)
    transactions = []
    known_ids = set()
    try:
        with filename.open("r", encoding="utf-8-sig", newline="") as stream:
            reader = csv.DictReader(stream)
            if reader.fieldnames != CSV_FIELDS:
                raise ValueError(f"Wrong CSV headers: expected {CSV_FIELDS}.")
            for line_no, row in enumerate(reader, start=2):
                try:
                    if None in row or any(row.get(field) is None for field in CSV_FIELDS):
                        raise ValueError("Missing or extra CSV columns")
                    record = Transaction(*(row[field] for field in CSV_FIELDS))
                    if record.transaction_id in known_ids:
                        raise ValueError("Duplicate transaction ID")
                    transactions.append(record)
                    known_ids.add(record.transaction_id)
                except (ValueError, TypeError) as exc:
                    print(f"Warning: skipped malformed CSV row {line_no}: {exc}")
    except FileNotFoundError:
        print(f"Missing CSV file: {filename}. Upload or restore it before continuing.")
        return []
    except (OSError, UnicodeError, csv.Error, ValueError) as exc:
        print(f"Could not read {filename}: {exc}")
        return []
    print(f"Loaded {len(transactions)} transaction(s) from {filename.name}.")
    return transactions


def save_transactions(transactions, filename):
    """Write atomically to a *working* CSV; refuse to overwrite source."""
    filename = Path(filename)
    if filename.resolve() == SAMPLE_CSV.resolve():
        print("Save blocked: the supplied sample CSV must not be overwritten.")
        return False
    temporary_path = None
    try:
        filename.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.NamedTemporaryFile(
            mode="w", encoding="utf-8", newline="", delete=False,
            dir=filename.parent, prefix=".tmp_finance_", suffix=".csv"
        ) as stream:
            temporary_path = Path(stream.name)
            writer = csv.DictWriter(stream, fieldnames=CSV_FIELDS)
            writer.writeheader()
            for transaction in transactions:
                writer.writerow(transaction.to_row())
        os.replace(temporary_path, filename)
        print(f"Saved {len(transactions)} transaction(s) to {filename.name}.")
        return True
    except (OSError, ValueError, TypeError) as exc:
        print(f"Save failed: {exc}")
        return False
    finally:
        if temporary_path is not None:
            try:
                temporary_path.unlink(missing_ok=True)
            except OSError:
                pass


def initialise_working_data(sample_path=SAMPLE_CSV, working_path=WORKING_CSV):
    """Prefer previous working data; seed from the original only once."""
    sample_path, working_path = Path(sample_path), Path(working_path)
    if working_path.exists():
        print("Restoring previous working data (not re-seeding from sample).")
        return load_transactions(working_path)
    if not sample_path.exists():
        print("No working file or sample found. Upload the sample/backup CSV.")
        return []
    records = load_transactions(sample_path)
    if records:
        save_transactions(records, working_path)
    return records



In [ ]:

from pathlib import Path

print("Original CSV exists:", SAMPLE_CSV.exists())
print("Working CSV exists:", WORKING_CSV.exists())

if WORKING_CSV.exists():
    restored = load_transactions(WORKING_CSV)
    print("Records restored:", len(restored))

    for transaction in restored:
        if transaction.transaction_id == "T010":
            print("Successfully restored:", transaction)


Original CSV exists: True
Working CSV exists: True
Loaded 9 transaction(s) from my_transactions.csv.
Records restored: 9
Successfully restored: T010 | 14/11/2026 | Food | Dinner | $-25.50


## 4 — Salted password verification
`PBKDF2-HMAC-SHA256` derives a one-way verifier from a password and random salt. Each classroom account is created in memory using an interactive password prompt. No fixed password appears in this notebook.

In [ ]:
def make_password_verifier(password):
    """Return PBKDF2 parameters + random salt + digest; never plaintext."""
    if not isinstance(password, str) or not password:
        raise ValueError("Password cannot be empty.")
    iterations = 260_000
    salt = secrets.token_bytes(16)
    derived = hashlib.pbkdf2_hmac("sha256", password.encode("utf-8"), salt,
                                 iterations)
    return f"pbkdf2_sha256${iterations}${salt.hex()}${derived.hex()}"


def verify_password(password, stored_verifier):
    """Recompute the PBKDF2 digest and compare in constant-time style."""
    try:
        algorithm, rounds, salt_hex, digest_hex = stored_verifier.split("$")
        if algorithm != "pbkdf2_sha256":
            return False
        iterations = int(rounds)
        if not 100_000 <= iterations <= 1_000_000:
            return False
        salt = bytes.fromhex(salt_hex)
        expected = bytes.fromhex(digest_hex)
        if len(salt) < 16 or len(expected) != 32:
            return False
        actual = hashlib.pbkdf2_hmac("sha256", password.encode("utf-8"),
                                    salt, iterations)
        return hmac.compare_digest(actual, expected)
    except (ValueError, TypeError, AttributeError):
        return False


def create_demo_accounts():
    """Temporary, in-memory accounts. Type passwords locally; do not save them."""
    print("Set temporary passwords for the fictional Admin and Viewer accounts.")
    accounts = {}
    for username in ("admin", "viewer"):
        while True:
            password = getpass(f"Create temporary password for {username}: ")
            if len(password) < 8:
                print("Use at least 8 characters for this demonstration.")
                continue
            accounts[username] = {"role": username,
                                  "verifier": make_password_verifier(password)}
            break
    print("In-memory verifiers ready; passwords are not stored in the notebook.")
    return accounts


def login(accounts):
    username = input("Username (admin/viewer): ").strip().lower()
    password = getpass("Password: ")
    details = accounts.get(username)
    if details is not None and verify_password(password, details["verifier"]):
        print(f"Login successful: {details['role']}")
        return details["role"]
    print("Login failed: invalid credentials.")
    return None


## 5 — Console menu
The Admin can Add/Display/Search/Delete; Viewer can Display/Search only, plus read-only totals. **To open the menu:** run `main()` in a new cell after the definitions and after restoring the CSV.

In [ ]:
def prompt_add_transaction(transactions, role):
    if not is_authorised(role, "add"):
        print(f"{role}: access denied for Add.")
        return False
    transaction_id = input("Transaction ID (e.g. T009): ").strip()
    date = input("Date (DD/MM/YYYY): ").strip()
    category = input("Category: ").strip()
    description = input("Description: ").strip()
    while True:
        amount = input("Amount (+income / -expense): ").strip()
        try:
            # Validate amount through the exact same model as the direct API.
            Transaction(transaction_id, date, category, description, amount)
            break
        except ValueError as exc:
            print(f"Input error: {exc}")
            if "Amount" not in str(exc):
                return False
    return add_transaction(transactions, transaction_id, date, category,
                           description, amount, role)


def main():
    transactions = initialise_working_data(SAMPLE_CSV, WORKING_CSV)
    if not SAMPLE_CSV.exists() and not WORKING_CSV.exists():
        print("Stopping to avoid creating an unexpected empty ledger.")
        return
    accounts = create_demo_accounts()

    while True:
        print("\n--- Sign in (enter Q as username to quit) ---")
        role = None
        username = input("Username (admin/viewer or Q): ").strip().lower()
        if username == "q":
            print("Tracker closed.")
            return
        password = getpass("Password: ")
        details = accounts.get(username)
        if details is not None and verify_password(password, details["verifier"]):
            role = details["role"]
            print(f"Login successful: {role}")
        else:
            print("Login failed: invalid credentials.")
            continue

        while True:
            print(f"\n=== PERSONAL FINANCE TRACKER | {role.upper()} ===")
            print("1. Add transaction" + (" (Admin only)" if role != "admin" else ""))
            print("2. Display transactions")
            print("3. Search by ID/category")
            print("4. Delete transaction" + (" (Admin only)" if role != "admin" else ""))
            print("5. Display category totals")
            print("6. Log out / switch account")
            print("7. Exit")
            choice = input("Choose 1-7: ").strip()
            if choice == "1":
                if prompt_add_transaction(transactions, role):
                    save_transactions(transactions, WORKING_CSV)
            elif choice == "2":
                display_transactions(transactions)
            elif choice == "3":
                query = input("Enter exact transaction ID or category keyword: ")
                search_transactions(transactions, query)
            elif choice == "4":
                if not is_authorised(role, "delete"):
                    print(f"{role}: access denied for Delete.")
                else:
                    transaction_id = input("Enter transaction ID to delete: ")
                    if delete_transaction(transactions, transaction_id, role):
                        save_transactions(transactions, WORKING_CSV)
            elif choice == "5":
                display_category_totals(transactions)
            elif choice == "6":
                print("Logged out.")
                break
            elif choice == "7":
                print("Goodbye. Existing working CSV is preserved.")
                return
            else:
                print("Invalid menu choice. Please enter a number from 1 to 7.")

# RUN THIS MANUALLY AFTER YOU UPLOAD OR MOUNT YOUR CSV:
# main()


## 6 — Functional evidence (run this cell)
This test performs an actual Admin addition, direct Viewer denials, invalid amount/date and duplicate-ID checks, CSV save/reload, salted-verifier tests and corrupted-row handling. It uses only the **synthetic** sample and temporary working files. These checks are reproducible, not a claim of a GitHub upload or a fresh cloud runtime.

In [ ]:
# Safe, repeatable demonstrations on fictional data in a temporary directory.
# No real passwords or financial details are printed or saved.
def run_evidence_tests():
    import io
    import contextlib
    from pathlib import Path
    from tempfile import TemporaryDirectory

    with TemporaryDirectory() as temp_folder:
        working = Path(temp_folder) / "test_working.csv"
        records = load_transactions(SAMPLE_CSV)
        assert len(records) == 8, "Expected the eight supplied synthetic rows."
        assert len(search_transactions(records, "Food")) == 3
        assert len(search_transactions(records, "T003")) == 1
        assert add_transaction(records, "T009", "10/11/2026", "Study",
                               "Textbook", -32.40, "admin")
        assert len(records) == 9
        assert not add_transaction(records, "T010", "10/11/2026", "Food",
                                   "Viewer trial", -3, "viewer")
        assert not delete_transaction(records, "T001", "viewer")
        assert len(records) == 9
        assert not add_transaction(records, "T009", "11/11/2026", "Food",
                                   "Duplicate", -2, "admin")
        assert not add_transaction(records, "T010", "11/11/2026", "Food",
                                   "Invalid input", "abc", "admin")
        assert not add_transaction(records, "T010", "31/02/2026", "Food",
                                   "Invalid date", -2, "admin")
        assert not delete_transaction(records, "T999", "admin")
        assert save_transactions(records, working)
        del records  # Drop in-memory data: must reload from the working file.
        restored = load_transactions(working)
        assert len(restored) == 9
        assert any(item.transaction_id == "T009" for item in restored)
        assert delete_transaction(restored, "T009", "admin")
        assert save_transactions(restored, working)
        restored_again = load_transactions(working)
        assert len(restored_again) == 8
        assert len(restored_again) == len(load_transactions(SAMPLE_CSV))
        assert not save_transactions(restored_again, SAMPLE_CSV)
        # Read a malformed CSV row: retain valid rows and report row number.
        invalid = Path(temp_folder) / "invalid.csv"
        invalid.write_text("transaction_id,date,category,description,amount\n"
                           "T099,10/11/2026,Food,Valid,-2.50\n"
                           "T100,31/02/2026,Food,Invalid,-1.00\n", encoding="utf-8")
        clean = load_transactions(invalid)
        assert len(clean) == 1
        assert load_transactions(Path(temp_folder) / "missing.csv") == []
        assert category_totals(restored_again)["Food"] == -69.20
        demo_secret = secrets.token_urlsafe(18)  # Generated in this run; never committed.
        verifier_a = make_password_verifier(demo_secret)
        verifier_b = make_password_verifier(demo_secret)
        assert verifier_a != verifier_b  # Independent random salts.
        assert verify_password(demo_secret, verifier_a)
        assert not verify_password(demo_secret + "!", verifier_a)
        assert is_authorised("admin", "delete")
        assert not is_authorised("viewer", "delete")
        assert not is_authorised("unknown", "add")
        print("EVIDENCE TEST RESULT: ALL CHECKS PASSED")
        print("Working CSV reloaded after discarding in-memory list: verified")
        print("Viewer direct Add/Delete blocked: verified")
        print("Sample CSV overwrite blocked: verified")

# Run in a fresh environment after uploading the original CSV.
run_evidence_tests()


Loaded 8 transaction(s) from MIT103_A3_Revised_T3_2026_Sample_Transactions.csv.
Found 3 result(s):
T001 | 02/11/2026 | Food | Groceries | $-48.50
T004 | 05/11/2026 | Food | Lunch | $-14.20
T008 | 09/11/2026 | Food | Coffee | $-6.50
Found 1 result(s):
T003 | 04/11/2026 | Income | Casual shift | $+180.00
Admin successfully added: T009
viewer: access denied for Add.
viewer: access denied for Delete.
Input error: duplicate transaction ID T009.
Input error: Amount must be a number, e.g. -12.50 or 200.00.
Input error: Date is not a real calendar date.
Delete failed: transaction T999 was not found.
Saved 9 transaction(s) to test_working.csv.
Loaded 9 transaction(s) from test_working.csv.
Deleted T009.
Saved 8 transaction(s) to test_working.csv.
Loaded 8 transaction(s) from test_working.csv.
Loaded 8 transaction(s) from MIT103_A3_Revised_T3_2026_Sample_Transactions.csv.
Save blocked: the supplied sample CSV must not be overwritten.
Loaded 1 transaction(s) from invalid.csv.
Missing CSV file: /t

In [ ]:

from pathlib import Path

sample_file = Path(
    "MIT103_A3_Revised_T3_2026_Sample_Transactions.csv"
)

original_data = """transaction_id,date,category,description,amount
T001,02/11/2026,Food,Groceries,-48.50
T002,03/11/2026,Transport,Train card,-25.00
T003,04/11/2026,Income,Casual shift,180.00
T004,05/11/2026,Food,Lunch,-14.20
T005,06/11/2026,Bills,Mobile plan,-35.00
T006,07/11/2026,Transport,Bus ticket,-5.60
T007,08/11/2026,Income,Refund,20.00
T008,09/11/2026,Food,Coffee,-6.50
"""

if not sample_file.exists():
    sample_file.write_text(original_data, encoding="utf-8")

print("Original sample exists:", sample_file.exists())
print("Sample records:", len(load_transactions(sample_file)))


Original sample exists: True
Loaded 8 transaction(s) from MIT103_A3_Revised_T3_2026_Sample_Transactions.csv.
Sample records: 8


In [ ]:
main()















Restoring previous working data (not re-seeding from sample).
Loaded 8 transaction(s) from my_transactions.csv.
Set temporary passwords for the fictional Admin and Viewer accounts.
Create temporary password for admin: ··········
Create temporary password for viewer: ··········
In-memory verifiers ready; passwords are not stored in the notebook.

--- Sign in (enter Q as username to quit) ---
Username (admin/viewer or Q): admin
Password: ··········
Login successful: admin

=== PERSONAL FINANCE TRACKER | ADMIN ===
1. Add transaction
2. Display transactions
3. Search by ID/category
4. Delete transaction
5. Display category totals
6. Log out / switch account
7. Exit
Choose 1-7: 1
Transaction ID (e.g. T009): T010
Date (DD/MM/YYYY): 14/11/2026
Category: Food
Description: Dinner
Amount (+income / -expense): -25.50
Admin successfully added: T010
Saved 9 transaction(s) to my_transactions.csv.

=== PERSONAL FINANCE TRACKER | ADMIN ===
1. Add transaction
2. Display transactions
3. Search by ID/cat

KeyboardInterrupt: Interrupted by user

In [ ]:

# Assessment 3: Viewer permission testing
# Test direct function access without changing saved data

test_transactions = load_transactions(WORKING_CSV)

print("\n=== VIEWER PERMISSION TEST ===")

print("\nTest 1: Viewer attempts to add a transaction")
add_result = add_transaction(
    test_transactions,
    "T099",
    "14/11/2026",
    "Food",
    "Test purchase",
    -10.00,
    "viewer"
)

print("\nTest 2: Viewer attempts to delete T001")
delete_result = delete_transaction(
    test_transactions,
    "T001",
    "viewer"
)

print("\n=== TEST SUMMARY ===")
print("Viewer Add allowed:", add_result)
print("Viewer Delete allowed:", delete_result)
print("Transactions remaining:", len(test_transactions))


Loaded 9 transaction(s) from my_transactions.csv.

=== VIEWER PERMISSION TEST ===

Test 1: Viewer attempts to add a transaction
viewer: access denied for Add.

Test 2: Viewer attempts to delete T001
viewer: access denied for Delete.

=== TEST SUMMARY ===
Viewer Add allowed: False
Viewer Delete allowed: False
Transactions remaining: 9


In [ ]:

from pathlib import Path

working_file = Path("my_transactions.csv")

print("Working CSV exists:", working_file.exists())

if working_file.exists():
    print("File size:", working_file.stat().st_size, "bytes")

    restored_transactions = load_transactions(working_file)

    print("Restored records:", len(restored_transactions))

    for transaction in restored_transactions:
        if transaction.transaction_id == "T010":
            print("Previously added transaction restored:")
            print(transaction)


Working CSV exists: True
File size: 406 bytes
Loaded 9 transaction(s) from my_transactions.csv.
Restored records: 9
Previously added transaction restored:
T010 | 14/11/2026 | Food | Dinner | $-25.50


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving my_transactions.csv to my_transactions.csv


In [ ]:
restored = load_transactions("my_transactions.csv")

print("Records restored:", len(restored))

for transaction in restored:
    if transaction.transaction_id == "T010":
        print("Successfully restored:", transaction)

NameError: name 'load_transactions' is not defined